# Práctica: clasificando reseñas de compradores

En `reseñas.csv` tienes 20 reseñas de un e-commerce, cada una con su etiqueta de sentimiento (`positiva` o `negativa`). Es un dataset chico a propósito: no se trata de lograr el mejor accuracy posible, sino de que construyas el pipeline completo con tus propias manos y entiendas qué hace cada paso.
 
Antes de escribir código, leé las 20 reseñas del CSV a mano. Fijate qué palabras aparecen repetidas en las positivas y cuáles en las negativas. Esa intuición te va a servir después para saber si el modelo está aprendiendo algo razonable o si está fallando por algo que vos ya podías anticipar.

## Ejercicio 1: cargar y explorar
 
Cargá el CSV con Pandas. Contá cuántas reseñas hay de cada clase (`positiva` / `negativa`). Esto no es un paso decorativo: si las clases estuvieran muy desbalanceadas (por ejemplo 18 positivas y 2 negativas), un modelo podría lograr un accuracy alto simplemente prediciendo siempre "positiva", sin haber aprendido nada útil. Antes de entrenar cualquier modelo de clasificación, siempre chequeá el balance de clases.

In [6]:
import pandas as pd

# Cargar el CSV
df = pd.read_csv("reseñas.csv")

## Ejercicio 2: tokenizar sin librerías
 
Escribí una función `tokenizar_simple(texto)` que reciba un string y devuelva una lista de palabras en minúscula, sin signos de puntuación, usando solo métodos nativos de Python (sin NLTK ni spaCy). Pista: puede que necesites la librería `string` y su lista de puntuación, además de `.split()`.
 
Después, aplicá `word_tokenize` de NLTK sobre la misma reseña y compará los resultados. ¿En qué casos tu función simple se equivoca o produce un resultado distinto al de NLTK? Escribí al menos un ejemplo concreto de una reseña del dataset donde la diferencia se note.

In [11]:
import pandas as pd
import string
import nltk
from nltk.tokenize import word_tokenize

# Cargar el dataset
df = pd.read_csv("reseñas.csv")

# Descargar recurso necesario para NLTK
nltk.download("punkt_tab")

# Función de tokenización simple
def tokenizar_simple(texto):
    texto = texto.lower()
    texto = texto.translate(
        str.maketrans("", "", string.punctuation)
    )
    return texto.split()

# Tomamos la primera reseña
reseña = df["texto"].iloc[0]

# Aplicamos las dos tokenizaciones
simple = tokenizar_simple(reseña)
nltk_tokens = word_tokenize(reseña)

# Mostrar resultados
print("RESEÑA ORIGINAL:")
print(reseña)

print("\n--- TOKENIZACIÓN SIMPLE ---")
print(simple)

print("\n--- TOKENIZACIÓN NLTK ---")
print(nltk_tokens)

print("\n--- CANTIDAD DE TOKENS ---")
print("Simple:", len(simple))
print("NLTK:", len(nltk_tokens))

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\comun\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


RESEÑA ORIGINAL:
el producto llegó en perfecto estado y antes de lo esperado

--- TOKENIZACIÓN SIMPLE ---
['el', 'producto', 'llegó', 'en', 'perfecto', 'estado', 'y', 'antes', 'de', 'lo', 'esperado']

--- TOKENIZACIÓN NLTK ---
['el', 'producto', 'llegó', 'en', 'perfecto', 'estado', 'y', 'antes', 'de', 'lo', 'esperado']

--- CANTIDAD DE TOKENS ---
Simple: 11
NLTK: 11


## Ejercicio 3: el efecto de las stopwords
 
Tomá la reseña `"no lo recomiendo, una pérdida de dinero total"` (o equivalente del dataset) y sacale las stopwords en español con NLTK. Mirá el resultado.
 
Ahora respondé sin correr más código, solo pensando: si esta reseña fuera parte de un modelo de Bag of Words que sacó stopwords, y la palabra "no" desapareció, ¿qué información se perdió? Buscá en el dataset si hay alguna otra reseña donde sacar "no" cambiaría el sentido de la frase. Este ejercicio no tiene una única respuesta "correcta" en código: el objetivo es que argumentes cuándo sacar stopwords ayuda y cuándo perjudica.

In [12]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download("stopwords")
nltk.download("punkt_tab")

reseña = "no lo recomiendo, una pérdida de dinero total"

# Tokenizar
tokens = word_tokenize(reseña)

# Stopwords en español
stopwords_es = set(stopwords.words("spanish"))

# Eliminar stopwords
tokens_sin_stopwords = [
    palabra for palabra in tokens
    if palabra.lower() not in stopwords_es
]

print("RESEÑA ORIGINAL:")
print(reseña)

print("\nTOKENS:")
print(tokens)

print("\nTOKENS SIN STOPWORDS:")
print(tokens_sin_stopwords)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\comun\AppData\Roaming\nltk_data...


RESEÑA ORIGINAL:
no lo recomiendo, una pérdida de dinero total

TOKENS:
['no', 'lo', 'recomiendo', ',', 'una', 'pérdida', 'de', 'dinero', 'total']

TOKENS SIN STOPWORDS:
['recomiendo', ',', 'pérdida', 'dinero', 'total']


[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\comun\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


## Ejercicio 4: vectorizar con TF-IDF
 
Usando `TfidfVectorizer` de sklearn, vectorizá las 20 reseñas del dataset completo (no hace falta separar en train/test todavía). Imprimí el vocabulario completo que generó el vectorizador con `.get_feature_names_out()`.
 
Buscá en ese vocabulario las 5 palabras con el IDF más alto (es decir, las más "raras" o distintivas del corpus) y las 5 con el IDF más bajo. Podés acceder a esos valores con el atributo `.idf_` del vectorizador, en el mismo orden que el vocabulario. ¿Las palabras con IDF alto te parecen informativas sobre el sentimiento de la reseña donde aparecen?

In [19]:
pip install scikit-learn

   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.3 MB 3.3 MB/s eta 0:00:03
   ------ --------------------------------- 1.3/8.3 MB 3.9 MB/s eta 0:00:02
   ----------- ---------------------------- 2.4/8.3 MB 4.0 MB/s eta 0:00:02
   --------------- ------------------------ 3.1/8.3 MB 4.0 MB/s eta 0:00:02
   ------------------ --------------------- 3.9/8.3 MB 4.1 MB/s eta 0:00:02
   ------------------------ --------------- 5.0/8.3 MB 4.1 MB/s eta 0:00:01
   --------------------------- ------------ 5.8/8.3 MB 4.1 MB/s eta 0:00:01
   ------------------------------ --------- 6.3/8.3 MB 4.1 MB/s eta 0:00:01
   ------------------------------ --------- 6.3/8.3 MB 4.1 MB/s eta 0:00:01
   ------------------------------ --------- 6.3/8.3 MB 4.1 MB/s eta 0:00:01
   ---------------------------------- ----- 7.1/8.3 MB 3.1 MB/s eta 0:00:01
   ---------------------------------------  8.1/8.3 MB 3.3 MB/s eta 0:00:01
   ----------------


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

# Crear el vectorizador
vectorizer = TfidfVectorizer()

# Vectorizar las reseñas
X = vectorizer.fit_transform(df["texto"])

# Obtener vocabulario
vocabulario = vectorizer.get_feature_names_out()

print("Vocabulario completo:")
print(vocabulario)

Vocabulario completo:
['abierto' 'antes' 'anuncia' 'atención' 'avisarme' 'bien' 'buen' 'buena'
 'buenísima' 'caja' 'calidad' 'cancelaron' 'coincide' 'color' 'como'
 'compra' 'comprar' 'con' 'conforme' 'cual' 'cumplió' 'dañado' 'de' 'del'
 'descripción' 'destruida' 'diez' 'dudas' 'el' 'empaquetado' 'en' 'encima'
 'envío' 'es' 'esperado' 'estado' 'estafa' 'excelente' 'expectativas'
 'experiencia' 'faltaban' 'familia' 'foto' 'fotos' 'funciona' 'incompleto'
 'inferior' 'instalación' 'justo' 'la' 'las' 'llegar' 'llegó' 'lo' 'mala'
 'material' 'mensajes' 'mes' 'mi' 'mis' 'mostraban' 'muy' 'nada' 'no'
 'nunca' 'paquete' 'para' 'pedido' 'pedí' 'perfecto' 'piezas' 'precio'
 'producto' 'prometido' 'publicada' 'pésima' 'que' 'quedé' 'rapidísimo'
 'recomendé' 'recomiendo' 'reembolso' 'relación' 'resolvió' 'respondió'
 'rompió' 'se' 'semana' 'sencilla' 'servicio' 'sin' 'superó' 'tal' 'tardó'
 'terrible' 'todas' 'total' 'totalmente' 'un' 'una' 'uso' 'vendedor'
 'vino' 'volvería' 'ya']


In [21]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

X = vectorizer.fit_transform(df["texto"])

vocabulario = vectorizer.get_feature_names_out()

tabla_idf = pd.DataFrame({
    "palabra": vocabulario,
    "idf": vectorizer.idf_
})

print("5 palabras con IDF más alto:")
print(tabla_idf.sort_values("idf", ascending=False).head(5))

print("\n5 palabras con IDF más bajo:")
print(tabla_idf.sort_values("idf", ascending=True).head(5))

5 palabras con IDF más alto:
    palabra       idf
0   abierto  3.351375
1     antes  3.351375
2   anuncia  3.351375
4  avisarme  3.351375
5      bien  3.351375

5 palabras con IDF más bajo:
   palabra       idf
28      el  2.098612
49      la  2.098612
52   llegó  2.252763
53      lo  2.252763
22      de  2.435085
